# Recommendation Engine

In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import joblib
import pandas as pd

In [2]:

features = [
    "has_hook", "has_face", "has_cta", "has_subtitle",
    "format", "text_density", "video_duration_s", "music_voice_ratio",
    "platform", "objective", "category", "target_audience_age", "is_retargeting",
    # Features de interação
    "hook_tiktok", "hook_linkedin", "hook_meta",
    "face_2534", "face_1824", "face_linkedin", "face_tiktok",
    "subtitle_linkedin",
    "creative_quality_score",
    "duration_fit",
]



# Métricas prioritárias por plataforma (para Camada 2)
PLATFORM_PRIORITY_METRIC = {
    "TikTok":   "ctr",
    "Meta":     "ctr",
    "LinkedIn": "roas",
}

# Features acionáveis e como simular a mudança
ACTIONABLE_FEATURES = {
    "has_hook":        {"type": "bool",     "change": 1,        "label": "Adicionar hook nos primeiros 3s"},
    "has_face":        {"type": "bool",     "change": 1,        "label": "Incluir rosto humano no criativo"},
    "has_cta":         {"type": "bool",     "change": 1,        "label": "Adicionar call-to-action"},
    "has_subtitle":    {"type": "bool",     "change": 1,        "label": "Adicionar legendas"},
    "format":          {"type": "cat",      "options": ["vertical", "horizontal", "quadrado"], "label": "Mudar formato para"},
    "text_density":    {"type": "cat",      "options": ["low", "medium", "high"], "label": "Ajustar densidade de texto para"},
    "video_duration_s":{"type": "duration", "buckets": {"≤15s": 12, "16-30s": 23, "31-45s": 38, "46-60s": 53}, "label": "Ajustar duração para"},
}

def preprocess_input(campanha: dict) -> pd.DataFrame:
    row = pd.DataFrame([campanha])
    
    # Conversão de tipos
    row["has_subtitle"] = row["has_subtitle"].map(
        {'True': 1, 'False': 0, 'unknown': 0, True: 1, False: 0}).astype(int)
    for col in ["has_hook", "has_face", "has_cta", "is_retargeting"]:
        row[col] = row[col].map({True: 1, False: 0, 1: 1, 0: 0}).astype(int)

    # Features de interação
    row["hook_tiktok"]   = ((row["has_hook"] == 1) & (row["platform"] == "TikTok")).astype(int)
    row["hook_linkedin"] = ((row["has_hook"] == 1) & (row["platform"] == "LinkedIn")).astype(int)
    row["hook_meta"]     = ((row["has_hook"] == 1) & (row["platform"] == "Meta")).astype(int)
    row["face_2534"]     = ((row["has_face"] == 1) & (row["target_audience_age"] == "25-34")).astype(int)
    row["face_1824"]     = ((row["has_face"] == 1) & (row["target_audience_age"] == "18-24")).astype(int)
    row["face_linkedin"] = ((row["has_face"] == 1) & (row["platform"] == "LinkedIn")).astype(int)
    row["face_tiktok"]   = ((row["has_face"] == 1) & (row["platform"] == "TikTok")).astype(int)
    row["subtitle_linkedin"] = ((row["has_subtitle"] == 1) & (row["platform"] == "LinkedIn")).astype(int)
    row["creative_quality_score"] = (
        row["has_hook"] + row["has_face"] + row["has_cta"] + row["has_subtitle"]
    )
    row["duration_fit"] = 0
    row.loc[(row["platform"] == "TikTok")   & (row["video_duration_s"].between(16, 30)), "duration_fit"] = 1
    row.loc[(row["platform"] == "Meta")     & (row["video_duration_s"] <= 15),            "duration_fit"] = 1
    row.loc[(row["platform"] == "LinkedIn") & (row["video_duration_s"] >= 60),            "duration_fit"] = 1

    return row[features]

def get_klike_score(campanha: dict) -> float:
    """Prediz o klike_score para uma campanha."""
    row = preprocess_input(campanha)
    return final_model.predict(row)[0]

In [3]:
def camada1_model(campanha: dict) -> list:
    """
    Simula mudanças em cada feature acionável e calcula
    o delta no klike_score. Retorna lista de candidatos.
    """
    score_atual = get_klike_score(campanha)
    candidatos = []

    for feat, config in ACTIONABLE_FEATURES.items():
        valor_atual = campanha.get(feat)

        if config["type"] == "bool":
            # Só recomenda se o atributo está ausente
            if valor_atual in [False, 0, "False"]:
                campanha_sim = campanha.copy()
                campanha_sim[feat] = True
                score_novo = get_klike_score(campanha_sim)
                delta = score_novo - score_atual
                if delta > 0:
                    candidatos.append({
                        "feature":     feat,
                        "change":      f"{feat}=1",
                        "label":       config["label"],
                        "delta_score": round(delta, 2),
                        "layer":       1,
                    })

        elif config["type"] == "cat":
            for opcao in config["options"]:
                if opcao == valor_atual:
                    continue
                campanha_sim = campanha.copy()
                campanha_sim[feat] = opcao
                score_novo = get_klike_score(campanha_sim)
                delta = score_novo - score_atual
                if delta > 0:
                    candidatos.append({
                        "feature":     feat,
                        "change":      f"{feat}={opcao}",
                        "label":       f"{config['label']} '{opcao}'",
                        "delta_score": round(delta, 2),
                        "layer":       1,
                    })

        elif config["type"] == "duration":
            for bucket_label, valor_sim in config["buckets"].items():
                if abs(valor_sim - valor_atual) < 5:
                    continue  # já está nessa faixa
                campanha_sim = campanha.copy()
                campanha_sim["video_duration_s"] = valor_sim
                score_novo = get_klike_score(campanha_sim)
                delta = score_novo - score_atual
                if delta > 0:
                    candidatos.append({
                        "feature":     feat,
                        "change":      f"duration={bucket_label}",
                        "label":       f"{config['label']} ({bucket_label})",
                        "delta_score": round(delta, 2),
                        "layer":       1,
                    })

    return candidatos

In [4]:
def camada2_lookup(campanha: dict) -> list:
    """
    Consulta a lookup table para a plataforma da campanha
    e retorna deltas nas métricas de performance.
    """
    platform = campanha["platform"]
    metric   = PLATFORM_PRIORITY_METRIC.get(platform, "ctr")
    candidatos = []

    for feat, config in ACTIONABLE_FEATURES.items():
        valor_atual = campanha.get(feat)

        # Define quais mudanças consultar
        if config["type"] == "bool":
            if valor_atual in [False, 0, "False"]:
                changes = [f"{feat}=1"]
            else:
                continue
        elif config["type"] == "cat":
            changes = [f"{feat}={op}" for op in config["options"] if op != valor_atual]
        elif config["type"] == "duration":
            changes = [f"duration={b}" for b in config["buckets"].keys()]
        else:
            changes = []

        for change in changes:
            row = lookup_table[
                (lookup_table["feature"]        == feat) &
                (lookup_table["change"]         == change) &
                (lookup_table["segment_type"]   == "platform") &
                (lookup_table["segment_value"]  == platform) &
                (lookup_table["metric"]         == metric)
            ]
            if row.empty:
                continue
            delta_pct = row["delta_pct"].values[0]
            if delta_pct > 0:
                label = config["label"]
                if config["type"] == "cat":
                    opcao = change.split("=")[1]
                    label = f"{config['label']} '{opcao}'"
                elif config["type"] == "duration":
                    bucket = change.replace("duration=", "")
                    label  = f"{config['label']} ({bucket})"

                candidatos.append({
                    "feature":    feat,
                    "change":     change,
                    "label":      label,
                    "delta_pct":  round(delta_pct, 1),
                    "metric":     metric,
                    "layer":      2,
                })

    return candidatos

In [5]:
def fusao_recomendacoes(candidatos_c1: list, candidatos_c2: list, top_n: int = 3) -> pd.DataFrame:

    df_c1 = pd.DataFrame(candidatos_c1) if candidatos_c1 else pd.DataFrame(
        columns=["feature", "change", "label", "delta_score", "score_norm"])
    df_c2 = pd.DataFrame(candidatos_c2) if candidatos_c2 else pd.DataFrame(
        columns=["feature", "change", "delta_pct", "metric", "score_norm"])

    # Normaliza Camada 1
    if not df_c1.empty and df_c1["delta_score"].max() > 0:
        df_c1["score_norm"] = df_c1["delta_score"] / df_c1["delta_score"].max()
    else:
        df_c1["score_norm"] = 0

    # Normaliza Camada 2
    if not df_c2.empty and df_c2["delta_pct"].max() > 0:
        df_c2["score_norm"] = df_c2["delta_pct"] / df_c2["delta_pct"].max()
    else:
        df_c2["score_norm"] = 0

    # Merge seguro
    if df_c1.empty and df_c2.empty:
        print("Nenhuma recomendação encontrada para essa campanha.")
        return pd.DataFrame()

    if df_c1.empty:
        merged = df_c2[["feature", "change", "delta_pct", "metric", "score_norm"]].rename(
            columns={"score_norm": "score_c2"})
        merged["delta_score"] = 0
        merged["score_c1"]    = 0
        merged["label"]       = merged["feature"]
    elif df_c2.empty:
        merged = df_c1[["feature", "change", "label", "delta_score", "score_norm"]].rename(
            columns={"score_norm": "score_c1"})
        merged["delta_pct"] = 0
        merged["metric"]    = ""
        merged["score_c2"]  = 0
    else:
        merged = pd.merge(
            df_c1[["feature", "change", "label", "delta_score", "score_norm"]].rename(
                columns={"score_norm": "score_c1"}),
            df_c2[["feature", "change", "delta_pct", "metric", "score_norm"]].rename(
                columns={"score_norm": "score_c2"}),
            on=["feature", "change"], how="outer"
        ).fillna(0)

    merged["score_final"] = 0.5 * merged["score_c1"] + 0.5 * merged["score_c2"]
    return merged.sort_values("score_final", ascending=False).head(top_n).reset_index(drop=True)

In [6]:
def gerar_recomendacoes(campanha: dict, top_n: int = 3) -> None:
    """Engine completo — recebe uma campanha e imprime as recomendações."""

    score_atual = get_klike_score(campanha)
    c1 = camada1_model(campanha)
    c2 = camada2_lookup(campanha)
    recomendacoes = fusao_recomendacoes(c1, c2, top_n=top_n)

    platform = campanha["platform"]
    metric   = PLATFORM_PRIORITY_METRIC.get(platform, "ctr")

    print("=" * 60)
    print(f"  RECOMMENDATIONS ENGINE — KLIKE")
    print("=" * 60)
    print(f"  Plataforma : {platform}")
    print(f"  Categoria  : {campanha.get('category', '—')}")
    print(f"  Objetivo   : {campanha.get('objective', '—')}")
    print(f"  klike_score atual: {score_atual:.1f} / 100")
    print("=" * 60)

    niveis = ["🔴 Alto impacto", "🟡 Médio impacto", "🟢 Impacto moderado"]

    for i, row in recomendacoes.iterrows():
        nivel = niveis[i] if i < len(niveis) else "💡 Sugestão"
        print(f"\n{nivel} — Recomendação {i+1}")

        label = row["label"] if row["label"] != 0 else row["feature"]
        print(f"  → {label}")

        if row["delta_score"] != 0:
            print(f"  • klike_score estimado: +{row['delta_score']:.1f} pontos")
        if row["delta_pct"] != 0:
            print(f"  • {metric.upper()} estimado: +{row['delta_pct']:.1f}% em {platform}")

    print("\n" + "=" * 60)

In [7]:
final_model = joblib.load("../models/final_model_fe.pkl")
print("Modelo carregado!")

lookup_table = pd.read_csv('../data/lookup_table.csv')


Modelo carregado!


In [8]:
# Widgets de input
w_platform  = widgets.Dropdown(options=["TikTok", "Meta", "LinkedIn"],
                                description="Plataforma:")
w_category  = widgets.Dropdown(options=["E-commerce", "SaaS", "App Install", "Branding", "Lead Gen"],
                                description="Categoria:")
w_objective = widgets.Dropdown(options=["awareness", "traffic", "conversions", "engagement", "app_install"],
                                description="Objetivo:")
w_age       = widgets.Dropdown(options=["18-24", "25-34", "35-44", "45+"],
                                description="Faixa etária:")
w_duration  = widgets.IntSlider(min=5, max=120, step=5, value=30,
                                 description="Duração (s):")
w_hook      = widgets.Checkbox(value=False, description="Tem hook?")
w_face      = widgets.Checkbox(value=False, description="Tem rosto?")
w_cta       = widgets.Checkbox(value=False, description="Tem CTA?")
w_subtitle  = widgets.Checkbox(value=False, description="Tem legenda?")
w_format    = widgets.Dropdown(options=["vertical", "horizontal", "quadrado"],
                                description="Formato:")
w_density   = widgets.Dropdown(options=["low", "medium", "high"],
                                description="Text density:")
w_ratio     = widgets.FloatSlider(min=0.0, max=1.0, step=0.1, value=0.5,
                                   description="Music/voice:")
w_retarget  = widgets.Checkbox(value=False, description="Retargeting?")
w_button    = widgets.Button(description="Gerar recomendações",
                              button_style="primary")
output      = widgets.Output()

def on_click(b):
    campanha = {
        "platform":           w_platform.value,
        "category":           w_category.value,
        "objective":          w_objective.value,
        "target_audience_age":w_age.value,
        "video_duration_s":   w_duration.value,
        "has_hook":           w_hook.value,
        "has_face":           w_face.value,
        "has_cta":            w_cta.value,
        "has_subtitle":       w_subtitle.value,
        "format":             w_format.value,
        "text_density":       w_density.value,
        "music_voice_ratio":  w_ratio.value,
        "is_retargeting":     w_retarget.value,
    }
    with output:
        clear_output()
        gerar_recomendacoes(campanha)

w_button.on_click(on_click)

ui = widgets.VBox([
    widgets.HBox([w_platform, w_category, w_objective]),
    widgets.HBox([w_age, w_format, w_density]),
    widgets.HBox([w_duration, w_ratio]),
    widgets.HBox([w_hook, w_face, w_cta, w_subtitle, w_retarget]),
    w_button,
    output
])

display(ui)